# Team Workbook — Day 5
## Prescriptive Analytics + Capstone  ·  Units IV & V

A prediction is not a decision. *"37 of these are likely to fail"* is interesting. *"Call these 12
first, in this order, and here is what it is worth"* is useful.

Today is the last mile: turning a model into something a person can act on tomorrow morning.

| Level | Question | You did it on |
|---|---|---|
| Descriptive | what happened? | Day 2 |
| Diagnostic | why did it happen? | Day 2 |
| Predictive | what will happen? | Days 3–4 |
| **Prescriptive** | **what should we do about it?** | **today** |

---


## Step 0 — Load everything back


In [ ]:
import pandas as pd, numpy as np, joblib
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)

FILE    = "day2_clean.csv"
NUM_COL = "____"
CAT_COL = "____"
TARGET  = None
TASK    = "auto"

df = pd.read_csv(FILE); df.columns = df.columns.str.strip()
df[NUM_COL] = pd.to_numeric(df[NUM_COL], errors="coerce")
print(df.shape)


### Paste the modelling kit


In [ ]:
"""
MLKIT  —  the modelling helpers used on Days 3, 4 and 5.
Written so that the SAME code works on every team's dataset.
Paste this whole cell once, near the top of your notebook.
"""
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score)

MAX_ONEHOT = 25          # a category with more levels than this is dropped
MAX_NULL_FRAC = 0.5      # a column emptier than this is dropped


# ---------------------------------------------------------------- target
def make_target(df, num_col=None, target=None, task="auto", verbose=True):
    """
    Returns (y, task, derived_from, label).
    target=None  -> derive a yes/no target by splitting num_col at its median.
                    This works on ANY dataset that has one number in it.
    """
    derived_from = None
    if target is None:
        if num_col is None:
            raise ValueError("Give either target= or num_col= so a target can be derived.")
        s = pd.to_numeric(df[num_col], errors="coerce")
        cut = s.median()
        y = (s >= cut).astype("float")
        y[s.isna()] = np.nan
        task, derived_from = "classification", num_col
        label = f"is {num_col} at or above its median ({cut:,.2f})?"
    else:
        s = df[target]
        if pd.api.types.is_numeric_dtype(s):
            nunique = s.nunique(dropna=True)
            if task == "auto":
                task = "classification" if nunique <= 10 else "regression"
            y = pd.to_numeric(s, errors="coerce")
        else:
            task = "classification" if task == "auto" else task
            levels = sorted(s.dropna().astype(str).unique())
            if len(levels) == 2:
                mapping = {levels[0]: 0, levels[1]: 1}
                y = s.astype(str).map(mapping)
                y[s.isna()] = np.nan
                print(f"ENCODED  : {levels[0]} -> 0    {levels[1]} -> 1  "
                      f"(so \"positive\" below means {levels[1]})")
            else:
                y = s.astype(str)
        label = f"{target}"

    if verbose:
        print(f"TARGET   : {label}")
        print(f"TASK     : {task}")
        if derived_from:
            print(f"           (derived from {derived_from} - so {derived_from} MUST be excluded from the features)")
        if task == "classification":
            vc = pd.Series(y).value_counts(dropna=True)
            share = (vc / vc.sum() * 100).round(1)
            print(f"BALANCE  : " + "  ".join(f"{k}={v}%" for k, v in share.items()))
            if share.max() > 80:
                print("           WARNING - one class dominates. Accuracy will look great and mean nothing.")
    return y, task, derived_from, label


# ---------------------------------------------------------------- features
def make_features(df, y, derived_from=None, target=None, drop=None, verbose=True):
    """One-hot encodes, imputes, and removes anything that would leak or be useless."""
    drop = set(drop or [])
    if derived_from: drop.add(derived_from)
    if target:       drop.add(target)

    X = df.copy()
    n = len(X)
    removed = []

    for c in list(X.columns):
        if c in drop:
            removed.append((c, "excluded on purpose - it would leak the answer" if c == derived_from
                               else "this is the target")); X = X.drop(columns=c); continue
        s = X[c]
        if s.isnull().mean() > MAX_NULL_FRAC:
            removed.append((c, f"{s.isnull().mean()*100:.0f}% empty")); X = X.drop(columns=c); continue
        if s.nunique(dropna=True) <= 1:
            removed.append((c, "same value in every row")); X = X.drop(columns=c); continue
        if pd.api.types.is_datetime64_any_dtype(s):
            removed.append((c, "a date - engineer month/weekday from it first")); X = X.drop(columns=c); continue
        if not pd.api.types.is_numeric_dtype(s):
            u = s.nunique(dropna=True)
            if u == n:
                looks_dated = pd.to_datetime(s, errors="coerce", format="mixed").notna().mean() > 0.8
                removed.append((c, "a date - engineer month/weekday from it first" if looks_dated
                                   else "one value per row - an ID, not a feature"))
                X = X.drop(columns=c); continue
            if u > MAX_ONEHOT:
                removed.append((c, f"{u} categories - too many to one-hot")); X = X.drop(columns=c); continue
        else:
            if pd.api.types.is_integer_dtype(s) and s.nunique(dropna=True) == n and n > 10:
                removed.append((c, "one value per row - an ID, not a feature")); X = X.drop(columns=c); continue

    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    for c in num_cols:
        if X[c].isnull().any():
            X[c] = X[c].fillna(X[c].median())
    for c in cat_cols:
        X[c] = X[c].astype(str).str.strip().fillna("missing")

    X = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=float)

    keep = pd.Series(y).notna().values
    X, y = X.loc[keep], pd.Series(y).loc[keep]

    if verbose:
        print(f"\nFEATURES : {X.shape[1]} columns built from {len(num_cols)} numeric + {len(cat_cols)} categorical")
        print(f"ROWS     : {len(X)} usable")
        if removed:
            print("REMOVED  :")
            for c, why in removed[:12]:
                print(f"           {c:<24} {why}")
    return X, y


# ---------------------------------------------------------------- split
def split(X, y, task, test_size=0.25, seed=42):
    strat = y if (task == "classification" and pd.Series(y).value_counts().min() >= 2) else None
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=strat)
    print(f"TRAIN    : {len(Xtr)} rows      TEST: {len(Xte)} rows  (the test rows are never used to fit)")
    return Xtr, Xte, ytr, yte


# ---------------------------------------------------------------- metrics
def score(y_true, y_pred, task, y_prob=None):
    if task == "regression":
        return {"MAE": mean_absolute_error(y_true, y_pred),
                "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
                "R2": r2_score(y_true, y_pred)}
    avg = "binary" if pd.Series(y_true).nunique() == 2 else "macro"
    out = {"Accuracy": accuracy_score(y_true, y_pred),
           "Precision": precision_score(y_true, y_pred, average=avg, zero_division=0),
           "Recall": recall_score(y_true, y_pred, average=avg, zero_division=0),
           "F1": f1_score(y_true, y_pred, average=avg, zero_division=0)}
    if y_prob is not None and pd.Series(y_true).nunique() == 2:
        try: out["ROC_AUC"] = roc_auc_score(y_true, y_prob)
        except Exception: pass
    return out


RESULTS = {}

def evaluate(name, model, Xtr, Xte, ytr, yte, task, store=True, verbose=True):
    """Fit, predict on the held-out test set, score, remember the result."""
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    prob = None
    if task == "classification" and hasattr(model, "predict_proba"):
        try: prob = model.predict_proba(Xte)[:, 1]
        except Exception: pass
    s = score(yte, pred, task, prob)
    if store: RESULTS[name] = s
    if verbose:
        print(f"{name:<22} " + "   ".join(f"{k} {v:.3f}" for k, v in s.items()))
    return model, s


def baseline(Xtr, Xte, ytr, yte, task):
    """The score to beat. A model that cannot beat this has learnt nothing."""
    m = DummyRegressor(strategy="mean") if task == "regression" else DummyClassifier(strategy="most_frequent")
    _, s = evaluate("BASELINE (no model)", m, Xtr, Xte, ytr, yte, task)
    return s


def leaderboard():
    if not RESULTS:
        print("Nothing evaluated yet."); return None
    t = pd.DataFrame(RESULTS).T.round(3)
    key = "R2" if "R2" in t.columns else "F1"
    return t.sort_values(key, ascending=False)


def cv(model, X, y, task, folds=5):
    scoring = "r2" if task == "regression" else "f1_macro"
    kf = KFold(folds, shuffle=True, random_state=42) if task == "regression" \
         else StratifiedKFold(folds, shuffle=True, random_state=42)
    s = cross_val_score(model, X, y, cv=kf, scoring=scoring)
    print(f"{folds}-fold {scoring}: " + " ".join(f"{v:.3f}" for v in s))
    print(f"  mean {s.mean():.3f}   spread +/- {s.std():.3f}")
    return s


def leak_check(X, y, task, thresh=0.95):
    """
    Flags any single feature that predicts the target almost perfectly on its own.
    The automatic guard only removes the column the target was DERIVED from.
    It cannot know that m1..m5 add up to total - only you can. This finds those.
    """
    yy = pd.Series(y).astype(float) if task == "regression" else pd.Series(y)
    suspects = []
    for c in X.columns:
        col = X[c]
        if col.nunique() <= 1:
            continue
        if task == "regression":
            r = abs(np.corrcoef(col, yy)[0, 1])
        else:
            try:    r = abs(np.corrcoef(col, pd.factorize(yy)[0])[0, 1])
            except Exception: continue
        if np.isfinite(r) and r >= thresh:
            suspects.append((c, round(float(r), 3)))
    if suspects:
        print("POSSIBLE LEAK - these features almost ARE the target:")
        for c, r in sorted(suspects, key=lambda t: -t[1]):
            print(f"   {c:<28} correlation {r}")
        print("   If a feature is computed FROM the target, drop it. A model that")
        print("   sees the answer is not predicting - it is reading.")
    else:
        print("No single feature gives the answer away. (Combinations still might.)")
    return suspects


def importances(model, X, top=12):
    if hasattr(model, "feature_importances_"):
        v = pd.Series(model.feature_importances_, index=X.columns)
    elif hasattr(model, "coef_"):
        c = np.ravel(model.coef_)
        v = pd.Series(np.abs(c), index=X.columns)
    else:
        print("This model does not expose importances."); return None
    return v.sort_values(ascending=False).head(top).round(4)


In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

y, task, derived_from, label = make_target(df, num_col=NUM_COL, target=TARGET, task=TASK)
X, y = make_features(df, y, derived_from=derived_from, target=TARGET)
X_train, X_test, y_train, y_test = split(X, y, task)

try:
    bundle = joblib.load("final_model.joblib")
    model  = bundle["model"]
    print(f"\nLoaded yesterday's model: {type(model).__name__}")
except Exception:
    RF = RandomForestRegressor if task == "regression" else RandomForestClassifier
    model = RF(n_estimators=200, random_state=42).fit(X_train, y_train)
    print("\nNo saved model found - refitted a random forest so you can carry on.")

_, s_final = evaluate("final model", model, X_train, X_test, y_train, y_test, task)


---
## Step 1 — What-if: how the prediction moves

Take one row. Change one thing. See what the model now predicts. This is how you turn a model into
advice — and it comes with a warning you must repeat out loud.


In [ ]:
row = X_test.iloc[[0]].copy()
numeric_feats = [c for c in X.columns if X[c].nunique() > 5]
FEATURE = numeric_feats[0] if numeric_feats else X.columns[0]

lo, hi = X[FEATURE].quantile(.05), X[FEATURE].quantile(.95)
sweep  = np.linspace(lo, hi, 25)

preds = []
for v in sweep:
    r = row.copy(); r[FEATURE] = v
    p = model.predict_proba(r)[0, 1] if (task == "classification" and hasattr(model, "predict_proba")) \
        else model.predict(r)[0]
    preds.append(p)

plt.figure(figsize=(7, 3.6))
plt.plot(sweep, preds, color="#2194D4", lw=2)
plt.axvline(row[FEATURE].iloc[0], color="#B4342F", ls="--", label="this row's actual value")
plt.xlabel(FEATURE)
plt.ylabel("predicted probability" if task == "classification" else f"predicted {NUM_COL}")
plt.title(f"If only {FEATURE} changed, holding everything else fixed")
plt.legend(); plt.tight_layout(); plt.show()

print(f"At the low end ({lo:,.2f}): {preds[0]:.3f}")
print(f"At the high end ({hi:,.2f}): {preds[-1]:.3f}")
print(f"Swing across the realistic range: {abs(preds[-1]-preds[0]):.3f}")


> **Say this in your presentation, in these words:** *"This shows how the model's prediction changes,
> not what would happen in the real world if we changed this."* The model learned from
> correlations. Nothing in your data proves that intervening on `FEATURE` would move the outcome.
> A curve like this is a **hypothesis worth testing**, not a result.

Every serious data scientist gets asked to forget this. Don't.


---
## Step 2 — The ranked action list

Nobody can act on everyone. Rank by predicted risk or predicted value, and act down the list until
the budget runs out. This is the single most useful artefact you will produce this week.


In [ ]:
full = df.loc[X.index].copy()

if task == "classification" and hasattr(model, "predict_proba"):
    full["score"] = model.predict_proba(X)[:, 1]
    score_name = "predicted probability"
else:
    full["score"] = model.predict(X)
    score_name = f"predicted {NUM_COL}"

full["rank"] = full["score"].rank(ascending=False, method="first").astype(int)

show = [c for c in [CAT_COL, NUM_COL, "score", "rank"] if c in full.columns]
print(f"TOP 15 by {score_name} - act on these first\n")
print(full.nlargest(15, "score")[show].round(3).to_string(index=False))

print(f"\n\nWhere the high scorers sit by {CAT_COL}:")
top20 = full.nlargest(max(int(len(full) * .2), 5), "score")
comp = pd.DataFrame({
    "top 20% (%)": top20[CAT_COL].value_counts(normalize=True).mul(100).round(1),
    "everyone (%)": full[CAT_COL].value_counts(normalize=True).mul(100).round(1),
})
comp["over/under"] = (comp["top 20% (%)"] - comp["everyone (%)"]).round(1)
print(comp.sort_values("over/under", ascending=False).to_string())


That last table is your targeting insight: a positive `over/under` means that category is
over-represented among the rows you would act on first.


---
## Step 3 — The cost of being wrong

`0.5` is not a magic number. It is a default, and defaults are rarely right.

A model flags someone. Two ways to be wrong:

- **False positive** — you act on someone who did not need it. You wasted the intervention.
- **False negative** — you missed someone who did. You lost whatever the outcome costs.

Those two almost never cost the same. Once you write down the two numbers, the correct threshold
falls out arithmetically instead of being guessed.


In [ ]:
# ===== EDIT THESE TWO NUMBERS FOR YOUR OWN SITUATION =====
COST_FALSE_ALARM = 1      # cost of acting on someone who did not need it
COST_MISS        = 10     # cost of failing to act on someone who did
# =========================================================

if task == "classification" and hasattr(model, "predict_proba"):
    prob = model.predict_proba(X_test)[:, 1]
    rows = []
    for t in np.arange(0.05, 0.96, 0.05):
        pred = (prob >= t).astype(int)
        yt   = np.asarray(y_test).astype(int)
        fp = int(((pred == 1) & (yt == 0)).sum())
        fn = int(((pred == 0) & (yt == 1)).sum())
        tp = int(((pred == 1) & (yt == 1)).sum())
        rows.append({"threshold": round(t, 2), "flagged": int(pred.sum()),
                     "caught": tp, "false alarms": fp, "missed": fn,
                     "total cost": fp * COST_FALSE_ALARM + fn * COST_MISS})
    costs = pd.DataFrame(rows)
    best  = costs.loc[costs["total cost"].idxmin()]

    print(costs.to_string(index=False))
    print(f"\nCheapest threshold: {best['threshold']}  (cost {int(best['total cost'])})")
    print(f"The default 0.5 would cost {int(costs.loc[costs.threshold==0.5,'total cost'].iloc[0])}")

    plt.figure(figsize=(7, 3.4))
    plt.plot(costs["threshold"], costs["total cost"], marker="o", color="#2194D4")
    plt.axvline(best["threshold"], color="#137A5E", ls="--", label=f"cheapest {best['threshold']}")
    plt.axvline(0.5, color="#B4342F", ls=":", label="default 0.5")
    plt.xlabel("threshold"); plt.ylabel("total cost"); plt.legend()
    plt.title("The threshold is a business decision, not a technical one")
    plt.tight_layout(); plt.show()
else:
    print("Regression: there is no threshold to choose. Skip to Step 4 -")
    print("your decision is how far down the ranked list you go, which is the same idea.")


**Change `COST_MISS` to 3 and re-run. Then to 50.** Watch the recommended threshold slide. Nothing
about the model changed — only what you said the mistakes were worth. That is prescriptive analytics
in one cell.


---
## Step 4 — Is the intervention worth doing at all?

Now put a number on the whole plan. Even rough numbers beat none — and state that they are rough.


In [ ]:
# ===== EDIT THESE FOR YOUR SITUATION =====
CAPACITY          = 30     # how many rows you can actually act on
COST_PER_ACTION   = 100    # what one intervention costs
VALUE_PER_SUCCESS = 900    # what one prevented/gained outcome is worth
SUCCESS_RATE      = 0.40   # share of correct flags where the action actually works
# =========================================

acted = full.nlargest(CAPACITY, "score")

if task == "classification":
    hit_rate = float(acted["score"].mean())              # model's own estimate
    expected_true = hit_rate * CAPACITY
else:
    thresh = full["score"].quantile(.75)
    expected_true = float((acted["score"] >= thresh).sum())
    hit_rate = expected_true / CAPACITY

spend  = CAPACITY * COST_PER_ACTION
gain   = expected_true * SUCCESS_RATE * VALUE_PER_SUCCESS
random_hits = (full["score"].mean() if task == "classification" else 0.5) * CAPACITY

print(f"Act on the top {CAPACITY} of {len(full)} rows")
print(f"  hit rate in that group     : {hit_rate:.1%}")
print(f"  if you picked at random    : {random_hits/CAPACITY:.1%}")
print(f"  the model's lift           : {hit_rate/max(random_hits/CAPACITY, 1e-9):.2f}x\n")
print(f"  spend                      : {spend:,.0f}")
print(f"  expected return            : {gain:,.0f}")
print(f"  net                        : {gain - spend:,.0f}")
print(f"  break-even success rate    : {spend / max(expected_true * VALUE_PER_SUCCESS, 1e-9):.1%}")
print("\nIf that break-even rate looks implausible, the honest recommendation is 'do not do this'.")


**The `lift` figure is the one to put on a slide.** *"Targeting with the model finds 2.4 times as
many as picking at random"* is a sentence a decision-maker understands immediately. Accuracy is not.


---
## Step 5 — Sensitivity: how wrong can you afford to be?


In [ ]:
rows = []
for sr in [0.1, 0.2, 0.3, 0.4, 0.6, 0.8]:
    g = expected_true * sr * VALUE_PER_SUCCESS
    rows.append({"if success rate is": f"{sr:.0%}", "return": round(g),
                 "net": round(g - spend), "worth doing": "yes" if g > spend else "no"})
pd.DataFrame(rows)


Find the row where "worth doing" flips. That is your margin of safety — and it belongs in your
presentation, because the first question any sensible person asks is *"what if you're wrong?"*


---
## Step 6 — Write the three recommendations

Not findings. **Recommendations.** Each one has an action, a target, an expected effect and a way to
tell whether it worked.

| | |
|---|---|
| **Action** | what specifically to do |
| **Who / what to apply it to** | from the ranked list in Step 2 |
| **Expected effect** | with the number from Step 4, and the confidence you actually have |
| **How you would know** | the measurement that would prove or disprove it |
| **What would make this wrong** | the assumption you are least sure of |

### Recommendation 1
- Action:
- Applies to:
- Expected effect:
- How we would know:
- What would make this wrong:

### Recommendation 2
- Action:
- Applies to:
- Expected effect:
- How we would know:
- What would make this wrong:

### Recommendation 3
- Action:
- Applies to:
- Expected effect:
- How we would know:
- What would make this wrong:

---


## Step 7 — Capstone assembly

Everything you need is now in five notebooks. This last cell pulls the headline numbers into one
place so you can copy them onto slides.


In [ ]:
print("="*66)
print("  CAPSTONE SUMMARY - copy these onto your slides")
print("="*66)
print(f"  Dataset          : {FILE}")
print(f"  Rows / features  : {len(X)} / {X.shape[1]}")
print(f"  Question         : {label}")
print(f"  Task             : {task}")
print(f"  Model            : {type(model).__name__}")
print(f"  Performance      : " + "  ".join(f"{k} {v:.3f}" for k, v in s_final.items()))
b = baseline(X_train, X_test, y_train, y_test, task)
print(f"  Baseline         : " + "  ".join(f"{k} {v:.3f}" for k, v in b.items()))
print(f"  Top 3 drivers    : {', '.join(importances(model, X, top=3).index)}"
      if importances(model, X, top=3) is not None else "")
print(f"  Recommended action on top {CAPACITY} rows, expected net {gain-spend:,.0f}")
print("="*66)


### Your eight slides

Ten minutes, four speakers, everybody talks. Nobody reads from the screen.

| # | Slide | Who | Must contain |
|---|---|---|---|
| 1 | The question | speaker 1 | Your dataset, where it came from, and the question in one sentence |
| 2 | The data, honestly | speaker 1 | Rows, columns, what was dirty, what you did about it, how much it moved the answer |
| 3 | What happened | speaker 2 | The summary table or one chart. Descriptive. |
| 4 | Why it happened | speaker 2 | The pattern, the effect size, and the reversal check. Diagnostic. |
| 5 | What will happen | speaker 3 | Target, features, the leakage you excluded, baseline vs model. Predictive. |
| 6 | How much to believe it | speaker 3 | Cross-validation spread, and the single biggest weakness |
| 7 | What to do | speaker 4 | The ranked list, the threshold, and the lift figure. Prescriptive. |
| 8 | What we would do next | speaker 4 | The one thing you would test, and what data you would need |

**Three rules that decide your mark more than anything else:**

1. Every number on a slide has an `n` beside it.
2. The word "causes" appears nowhere unless you ran an experiment. You did not.
3. Slide 6 is not optional. A team that cannot name its own biggest weakness has not understood
   its own work — and that is the question you will be asked individually.


---
## Final checklist

- [ ] All five notebooks run clean under `Runtime > Restart and run all`
- [ ] `final_model.joblib` and `model_card.txt` are in the folder
- [ ] Every recommendation in Step 6 has all five fields filled
- [ ] The eight slides exist, and every team member owns two of them
- [ ] Somebody in the team can answer: *"why did you choose that threshold?"*
- [ ] Somebody in the team can answer: *"what would change your conclusion?"*
